# Recommender Systems: Content-Based and Collaborative Filtering

In [2]:
import pandas as pd
import numpy as np

movies = pd.read_csv("data/movielens/movies.csv")
ratings = pd.read_csv("data/movielens/ratings.csv")

movies.head()
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


### Preprocessing genres

In [3]:
movies['genres'] = movies['genres'].str.replace('|', ' ')

---

## PART-1: Content-Based Filtering

### TASK-1: TF-IDF Based Recommendation

In [13]:
# TF-IDF Movie Representation (VECTORS)

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['genres'])

In [6]:
# computing cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [12]:
# Movie recommendation function

indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

def recommend_movies(title, top_n=5):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]
    results = movies[['title']].iloc[movie_indices].copy()
    results['similarity_score'] = [i[1] for i in sim_scores]

    return results


In [10]:
recommend_movies("Toy Story (1995)")

,title,similarity_score
1706,Antz (1998),1.0
2355,Toy Story 2 (1999),1.0
2809,"Adventures of Rocky and Bullwinkle, The (2000)",1.0
3000,"Emperor's New Groove, The (2000)",1.0
3568,"Monsters, Inc. (2001)",1.0


---

### TASK-2: User Profile Based Content Recommender

In [15]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
# mapping movieID to TF-IDF index 
movie_id_to_index = pd.Series(movies.index, index=movies.movieId)

In [17]:
def build_user_profile(user_id):

    # Movies rated by the user
    user_ratings = ratings[ratings['userId'] == user_id]

    # Get movie indices
    movie_indices = user_ratings['movieId'].map(movie_id_to_index)

    # TF-IDF vectors of those movies
    movie_vectors = tfidf_matrix[movie_indices]

    # Ratings as weights
    weights = user_ratings['rating'].values

    # Weighted average
    user_profile = np.average(movie_vectors.toarray(), axis=0, weights=weights)

    return user_profile

def recommend_movies_user(user_id, top_n=10):
    user_profile = build_user_profile(user_id)

    # Compute similarity with all movies
    similarity_scores = cosine_similarity([user_profile], tfidf_matrix)
    similarity_scores = similarity_scores.flatten()

    # Get movie indices sorted by similarity
    movie_indices = np.argsort(similarity_scores)[::-1]

    # Remove movies already rated
    rated_movies = ratings[ratings.userId == user_id]['movieId']
    rated_indices = rated_movies.map(movie_id_to_index)
    movie_indices = [i for i in movie_indices if i not in rated_indices]

    # Top recommendations
    top_movies = movie_indices[:top_n]
    recommendations = movies.iloc[top_movies][['title']].copy()
    recommendations['similarity_score'] = similarity_scores[top_movies]

    return recommendations

In [18]:
recommend_movies_user(1, top_n=5)

,title,similarity_score
8597,Dragonheart 2: A New Beginning (2000),0.791026
6570,"Hunting Party, The (2007)",0.773645
4005,Flashback (1990),0.773089
4681,The Great Train Robbery (1978),0.773089
5471,"Diamond Arm, The (Brilliantovaya ruka) (1968)",0.760503


---

## PART-2: Collaborative Filtering

### TASK-3: User-Based Collaborative Filtering

In [22]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv("data/movielens/movies.csv")
ratings = pd.read_csv("data/movielens/ratings.csv")

In [23]:
# user-movie rating matrix 
user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

In [24]:
# computing user similarity matrix 
user_similarity = cosine_similarity(user_movie_matrix.fillna(0))

#converting to dataframe 
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

In [25]:
def get_similar_users(user_id, k=5):
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)

    # remove self similarity
    similar_users = similar_users.drop(user_id)

    return similar_users.head(k)

In [31]:
print("Top similar users to User 1:")
get_similar_users(1)

Top similar users to User 1:


userId
266    0.357408
313    0.351562
368    0.345127
57     0.345034
91     0.334727
Name: 1, dtype: float64

In [27]:
def predict_rating(user_id, movie_id, k=5):
    similar_users = get_similar_users(user_id, k)
    numerator = 0
    denominator = 0

    for other_user, similarity in similar_users.items():
        rating = user_movie_matrix.loc[other_user, movie_id]
        if not np.isnan(rating):
            numerator += similarity * rating
            denominator += abs(similarity)

    if denominator == 0:
        return 0

    return numerator / denominator

In [28]:
def recommend_movies_user_cf(user_id, top_n=10):
    user_ratings = user_movie_matrix.loc[user_id]
    unrated_movies = user_ratings[user_ratings.isna()].index
    predicted_ratings = {}

    for movie_id in unrated_movies:
        predicted_ratings[movie_id] = predict_rating(user_id, movie_id)

    sorted_movies = sorted(
        predicted_ratings.items(),
        key=lambda x: x[1],
        reverse=True
    )

    top_movies = sorted_movies[:top_n]
    movie_ids = [i[0] for i in top_movies]
    recommendations = movies[movies.movieId.isin(movie_ids)][['title']]
    recommendations['predicted_rating'] = [i[1] for i in top_movies]

    return recommendations

In [29]:
recommend_movies_user_cf(1,5)

,title,predicted_rating
449,"Ref, The (1994)",5.0
474,Blade Runner (1982),5.0
585,Wallace & Gromit: The Best of Aardman Animatio...,5.0
602,Dr. Strangelove or: How I Learned to Stop Worr...,5.0
659,"Godfather, The (1972)",5.0


In [45]:
# Evaluation pipeline 

# 1. train-test split
from sklearn.model_selection import train_test_split
train_ratings, test_ratings = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

# 2. build user-movie matrix for training data
train_matrix = train_ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

# 3. compute user similarity on training data
train_matrix = train_ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

# 4. rating prediction function 
def predict_rating(user_id, movie_id, k=5):
    if movie_id not in train_matrix.columns:
        return np.nan

    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:k+1]
    numerator = 0
    denominator = 0

    for other_user, sim in similar_users.items():
        if movie_id in train_matrix.columns:
            rating = train_matrix.loc[other_user, movie_id]
            if not np.isnan(rating):
                numerator += sim * rating
                denominator += abs(sim)

    if denominator == 0:
        return np.nan

    return numerator / denominator


# 5. RMSE evaluation 
from sklearn.metrics import mean_squared_error

def compute_rmse():
    predictions = []
    actuals = []

    for row in test_ratings.itertuples():
        pred = predict_rating(row.userId, row.movieId)
        if not np.isnan(pred):
            predictions.append(pred)
            actuals.append(row.rating)

    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    return rmse

compute_rmse()

# 6. recommendation function using train matrix
def recommend_movies_cf(user_id, k=10):
    user_ratings = train_matrix.loc[user_id]
    unrated_movies = user_ratings[user_ratings.isna()].index[:200]
    predictions = {}
    for movie in unrated_movies:
        pred = predict_rating(user_id, movie)
        if not np.isnan(pred):
            predictions[movie] = pred

    top_movies = sorted(predictions.items(), key=lambda x: x[1], reverse=True)[:k]
    return [i[0] for i in top_movies]

# 7. Precision@K evaluation
def precision_at_k(user_id, k=5):

    recommended = recommend_movies_cf(user_id, k)

    relevant = test_ratings[
        (test_ratings.userId == user_id) &
        (test_ratings.rating >= 4)
    ]['movieId']

    hits = len(set(recommended) & set(relevant))

    return hits / k

# 8. Recall@K evaluation
def recall_at_k(user_id, k=5):

    recommended = recommend_movies_cf(user_id, k)

    relevant = test_ratings[
        (test_ratings.userId == user_id) &
        (test_ratings.rating >= 4)
    ]['movieId']

    if len(relevant) == 0:
        return 0

    hits = len(set(recommended) & set(relevant))

    return hits / len(relevant)


# 9. Evaluation all users
def evaluate_system(k=5):

    precisions = []
    recalls = []

    users = test_ratings.userId.unique()

    for user in users:

        if user not in train_matrix.index:
            continue

        p = precision_at_k(user, k)
        r = recall_at_k(user, k)

        precisions.append(p)
        recalls.append(r)

    print("Average Precision@{}: {:.3f}".format(k, np.mean(precisions)))
    print("Average Recall@{}: {:.3f}".format(k, np.mean(recalls)))
    

print("Evaluation Results")
print("------------------")

print("RMSE:", compute_rmse())

evaluate_system(5)

Evaluation Results
------------------
RMSE: 1.0595302481814641
Average Precision@5: 0.053
Average Recall@5: 0.032


---

### TASK-4: Item-Based Collaborative Filtering

In [55]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv("data/movielens/movies.csv")
ratings = pd.read_csv("data/movielens/ratings.csv")

In [56]:
#computing user-movie matrix
user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

In [57]:
# computing item similarity matrix
item_similarity = cosine_similarity(user_movie_matrix.T.fillna(0))

user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_movie_matrix.columns,
    columns=user_movie_matrix.columns
)

In [58]:
# prediction function
def predict_rating_item(user_id, movie_id):
    if movie_id not in item_similarity_df.index:
        return np.nan

    user_ratings = user_movie_matrix.loc[user_id].dropna()
    numerator = 0
    denominator = 0

    for rated_movie, rating in user_ratings.items():
        similarity = item_similarity_df.loc[movie_id, rated_movie]
        numerator += similarity * rating
        denominator += abs(similarity)

    if denominator == 0:
        return np.nan

    return numerator / denominator

#recommendation function
def recommend_movies_item_cf(user_id, top_n=10):
    user_ratings = user_movie_matrix.loc[user_id]
    unrated_movies = user_ratings[user_ratings.isna()].index
    predictions = {}

    for movie_id in unrated_movies:
        pred = predict_rating_item(user_id, movie_id)
        if not np.isnan(pred):
            predictions[movie_id] = pred

    top_movies = sorted(
        predictions.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_n]

    movie_ids = [i[0] for i in top_movies]
    recommendations = movies[movies.movieId.isin(movie_ids)][['title']]
    recommendations['predicted_rating'] = [i[1] for i in top_movies]
    return recommendations

In [59]:
#testing
recommend_movies_item_cf(1,5)

,title,predicted_rating
865,Entertaining Angels: The Dorothy Day Story (1996),5.0
1156,Broken English (1996),5.0
2179,Alvarez Kelly (1966),5.0
2348,Come See the Paradise (1990),5.0
2909,Circus (2000),5.0


In [ ]:
# Evaluation pipeline for item-based CF

from sklearn.model_selection import train_test_split

# Train-test split
train_ratings_item, test_ratings_item = train_test_split(
    ratings, test_size=0.2, random_state=42
)

# Build user-movie matrix from training data only
train_matrix_item = train_ratings_item.pivot_table(
    index='userId', columns='movieId', values='rating'
)

# Recompute item similarity on training data
item_similarity_train = cosine_similarity(train_matrix_item.T.fillna(0))
item_similarity_train_df = pd.DataFrame(
    item_similarity_train,
    index=train_matrix_item.columns,
    columns=train_matrix_item.columns
)

def predict_rating_item_eval(user_id, movie_id):
    if movie_id not in item_similarity_train_df.index:
        return np.nan
    if user_id not in train_matrix_item.index:
        return np.nan

    user_ratings = train_matrix_item.loc[user_id].dropna()
    numerator = 0
    denominator = 0

    for rated_movie, rating in user_ratings.items():
        if rated_movie not in item_similarity_train_df.columns:
            continue
        similarity = item_similarity_train_df.loc[movie_id, rated_movie]
        numerator += similarity * rating
        denominator += abs(similarity)

    if denominator == 0:
        return np.nan

    return numerator / denominator


def recommend_movies_item_cf_eval(user_id, top_n=10):
    if user_id not in train_matrix_item.index:
        return pd.DataFrame(columns=['title', 'predicted_rating'])

    user_ratings = train_matrix_item.loc[user_id]
    # Only recommend from movies NOT in training set for this user
    unrated_movies = user_ratings[user_ratings.isna()].index
    predictions = {}

    for movie_id in unrated_movies:
        pred = predict_rating_item_eval(user_id, movie_id)
        if not np.isnan(pred):
            predictions[movie_id] = pred

    top_movies = sorted(predictions.items(), key=lambda x: x[1], reverse=True)[:top_n]
    movie_ids = [i[0] for i in top_movies]
    recommendations = movies[movies.movieId.isin(movie_ids)][['title', 'movieId']].copy()
    recommendations['predicted_rating'] = recommendations['movieId'].map(dict(top_movies))
    return recommendations


def precision_at_k_item(user_id, k=5):
    recs = recommend_movies_item_cf_eval(user_id, k)
    recommended_ids = set(recs['movieId'].values)

    # Relevant = movies rated >= 4 in the TEST set (held-out)
    relevant_movies = set(test_ratings_item[
        (test_ratings_item.userId == user_id) & (test_ratings_item.rating >= 4)
    ].movieId)

    if len(recommended_ids) == 0:
        return 0

    hits = len(recommended_ids & relevant_movies)
    return hits / k


def recall_at_k_item(user_id, k=5):
    recs = recommend_movies_item_cf_eval(user_id, k)
    recommended_ids = set(recs['movieId'].values)

    # Relevant = movies rated >= 4 in the TEST set (held-out)
    relevant_movies = set(test_ratings_item[
        (test_ratings_item.userId == user_id) & (test_ratings_item.rating >= 4)
    ].movieId)

    if len(relevant_movies) == 0:
        return 0

    hits = len(recommended_ids & relevant_movies)
    return hits / len(relevant_movies)


def evaluate_item_cf(k=5):
    precisions = []
    recalls = []

    users = test_ratings_item.userId.unique()[:50]

    for user in users:
        if user not in train_matrix_item.index:
            continue

        precisions.append(precision_at_k_item(user, k))
        recalls.append(recall_at_k_item(user, k))

    print("Average Precision@{}: {:.6f}".format(k, np.mean(precisions)))
    print("Average Recall@{}: {:.6f}".format(k, np.mean(recalls)))


print("Item-Based CF Evaluation")
print("-----------------------")

evaluate_item_cf(5)


Item-Based CF Evaluation
-----------------------
Average Precision@5: 0.000
Average Recall@5: 0.000


---

### Comparing the user based and item based collaborative filtering approaches

---